In [42]:
import yfinance as yf
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path

## Walk-forward split

In [43]:
def walk_forward_splits(dates, first_val_year=2012, last_val_year=2019, gap=5):
    """Return a list of (train_index, val_index), one pair per validation year."""

    # 1. Distinct trading days, sorted from oldest to newest
    trading_days = list(pd.Series(dates.unique()).sort_values())

    splits = []

    # 2. One fold per validation year
    for year in range(first_val_year, last_val_year + 1):

        # 3. Find the position of the first trading day of this year
        pos = 0
        for i in range(len(trading_days)):
            if trading_days[i].year == year:
                pos = i
                break

        # 4. Last training day: go back gap + 1 trading days
        last_train_day = trading_days[pos - (gap + 1)]

        # 5. Go through every row and decide: train, validation, or ignored
        train_index = []
        val_index = []
        for row_number, date in dates.items():
            if date <= last_train_day:
                train_index.append(row_number)
            elif date.year == year:
                val_index.append(row_number)

        splits.append((train_index, val_index))

    return splits

In [44]:
df = pd.read_csv(
    "../data/processed/features.csv",
    parse_dates=["date"]
)

# Keep only 2006-2019
df = df[
    (df["date"] >= "2006-01-01") &
    (df["date"] <= "2019-12-31")
]

df = df.reset_index(drop=True)
dates = df["date"]

splits = walk_forward_splits(dates)

print(len(splits))

for split in splits:
    train_index, val_index = split
    print(len(train_index), len(val_index))

8
49698 8250
57948 8316
66264 8316
74580 8316
82896 8316
91212 8283
99495 8283
107778 8316


## Fonctions de métriques

In [45]:
from sklearn.metrics import (
    mean_squared_error,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix,
    accuracy_score,
)

In [46]:
def regression_metrics(y_true, y_pred):
    """
    Calcule les métriques de régression MSE et QLIKE entre les valeurs réelles et prédites.
    """

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Calcul MSE
    mse = mean_squared_error(y_true, y_pred)

    # Calcul variance
    sigma2 = np.exp(2 * y_true)
    h = np.exp(2 * y_pred)

    # Pour éviter la division par zero
    h = np.maximum(h, 1e-12)

    # perte QLIKE
    ratio = sigma2 / h
    qlike = np.mean(ratio - np.log(ratio) - 1)

    return {
        "MSE": mse,
        "QLIKE": qlike,
    }


In [47]:
def classification_metrics(y_true, proba, threshold=0.5, c_fn=5, c_fp=1):
    """
    Calcule les métriques de classification pour la détection des spikes de volatilité.
    """

    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    # Convertir les probas en prédictions binaires selon le seuil
    y_pred = []
    for p in proba:
        if p >= threshold:
            y_pred.append(1)
        else:
            y_pred.append(0)

    # matrice de confusion
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    # métriques de classification
    pr_auc = average_precision_score(y_true, proba)

    recall = recall_score(
        y_true, y_pred, zero_division=0
    )

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    # cout des faux positifs et faux négatifs
    cost = c_fn * fn + c_fp * fp

    accuracy = accuracy_score(y_true, y_pred)

    return {
        "PR_AUC": pr_auc,
        "Recall": recall,
        "Precision": precision,
        "F1": f1,
        "Cost": cost,
        "Accuracy": accuracy,
        "FN": fn,
        "FP": fp,
    }

## Baselines de classification

In [48]:
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# from src.evaluation import classification_metrics

In [49]:
features = [
    "log_vol_d",
    "log_vol_w",
    "log_vol_m"
]

X = df[features]
y = df["target_spike"]

print(X.shape, y.shape)
print(X.head())
print(y.head())

(116259, 3) (116259,)
   log_vol_d  log_vol_w  log_vol_m
0  -4.318065  -4.307015  -4.341245
1  -4.279739  -4.272838  -4.354880
2  -4.577694  -4.290378  -4.389434
3  -4.008196  -4.206642  -4.360492
4  -4.396927  -4.298690  -4.366826
0    0
1    0
2    0
3    0
4    1
Name: target_spike, dtype: int64


In [50]:
model_majority = DummyClassifier(
    strategy="most_frequent"
)

In [51]:
model_logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [67]:
results = []

models = {
    "Majority": model_majority,
    "Logistic": model_logistic
}

splits = walk_forward_splits(df["date"])

seuils = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

fold = 1

for split in splits:

    train_index = split[0]
    val_index = split[1]

    X_train = X.iloc[train_index]
    X_val = X.iloc[val_index]

    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        proba = model.predict_proba(X_val)[:, 1]

        if model_name == "Logistic":

            for seuil in seuils:

                metrics = classification_metrics(
                    y_val,
                    proba,
                    threshold=seuil
                )

                metrics["Fold"] = fold
                metrics["Model"] = model_name
                metrics["Seuil"] = seuil

                results.append(metrics)

        else:

            metrics = classification_metrics(
                y_val,
                proba
            )

            metrics["Fold"] = fold
            metrics["Model"] = model_name
            metrics["Seuil"] = 0.5

            results.append(metrics)

    fold = fold + 1

In [68]:
results = pd.DataFrame(results)

results_logistic = results[
    results["Model"] == "Logistic"
]

resume_seuils = results_logistic.groupby("Seuil").agg(
    {
        "Recall": "mean",
        "Precision": "mean",
        "F1": "mean",
        "Cost": "mean",
        "Accuracy": "mean"
    }
)

print(resume_seuils)

         Recall  Precision        F1      Cost  Accuracy
Seuil                                                   
0.05   0.824853   0.124084  0.211236  5563.875  0.394482
0.10   0.507812   0.182869  0.260611  3837.750  0.724235
0.20   0.202556   0.318584  0.237450  3603.875  0.879303
0.30   0.084426   0.444605  0.138492  3874.000  0.898871
0.40   0.033751   0.557101  0.062611  4045.125  0.900844
0.50   0.015320   0.717016  0.029632  4132.375  0.900030


In [70]:
results_df = pd.DataFrame(results)

print(results_df)

      PR_AUC    Recall  Precision        F1  Cost  Accuracy    FN    FP  Fold  \
0   0.040970  0.000000   0.000000  0.000000  1690  0.959030   338     0     1   
1   0.080724  0.801775   0.049688  0.093577  5518  0.363636    67  5183     1   
2   0.080724  0.411243   0.066988  0.115209  2931  0.741212   199  1936     1   
3   0.080724  0.121302   0.113573  0.117310  1805  0.925212   297   320     1   
4   0.080724  0.035503   0.137931  0.056471  1705  0.951394   326    75     1   
5   0.080724  0.011834   0.181818  0.022222  1688  0.957333   334    18     1   
6   0.080724  0.008876   0.375000  0.017341  1680  0.958788   335     5     1   
7   0.098244  0.000000   0.000000  0.000000  4085  0.901756   817     0     2   
8   0.192808  0.738066   0.126177  0.215511  5246  0.472102   214  4176     2   
9   0.192808  0.350061   0.180670  0.238333  3952  0.780183   531  1297     2   
10  0.192808  0.104039   0.314815  0.156394  3845  0.889731   732   185     2   
11  0.192808  0.036720   0.4

In [71]:
metrics_columns = [
    "PR_AUC",
    "Recall",
    "Precision",
    "F1",
    "Cost",
    "Accuracy"
]

# Résumé par modèle et par seuil
summary = results_df.groupby(["Model", "Seuil"])[
    metrics_columns
].agg(["mean", "std"])

print(summary)


                  PR_AUC              Recall           Precision            \
                    mean       std      mean       std      mean       std   
Model    Seuil                                                               
Logistic 0.05   0.238616  0.124164  0.824853  0.070915  0.124084  0.061749   
         0.10   0.238616  0.124164  0.507812  0.130113  0.182869  0.090209   
         0.20   0.238616  0.124164  0.202556  0.091596  0.318584  0.144125   
         0.30   0.238616  0.124164  0.084426  0.045195  0.444605  0.181349   
         0.40   0.238616  0.124164  0.033751  0.026186  0.557101  0.221934   
         0.50   0.238616  0.124164  0.015320  0.014121  0.717016  0.224884   
Majority 0.50   0.101385  0.052832  0.000000  0.000000  0.000000  0.000000   

                      F1                Cost               Accuracy            
                    mean       std      mean          std      mean       std  
Model    Seuil                                             

In [72]:
results_df.to_csv(
    "../results/baselines_classification.csv",
    index=False
)